# NLP Sentiment Analysis - Training File

Business objective: extract sentiment from ecommerce customer reviews using the provided `dataset -P676 (1).xlsx` file.

In [1]:
from pathlib import Path
import json
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATASET_PATH = PROJECT_ROOT / 'data' / 'raw' / 'dataset -P676 (1).xlsx'
print(f'Project root: {PROJECT_ROOT}')
print(f'Dataset: {DATASET_PATH.name}')

Project root: /Users/sai/Documents/exclr project
Dataset: dataset -P676 (1).xlsx


## 1. Load Dataset

In [2]:
raw_df = pd.read_excel(DATASET_PATH)
print(f'Rows: {raw_df.shape[0]}')
print(f'Columns: {raw_df.shape[1]}')
print(raw_df.head().to_string(index=False))

Rows: 1440
Columns: 3
                                  title  rating                                                                                                                                                                                                                                                                                                                                                                                                                body
                       Horrible product       1                                                                                                                                                                                                                                                                                                                                                         Very disappointed with the overall performance from Samsung
Camera quality is not like 48 megapixel       3                           

## 2. Prepare Text And Labels

Rating labels are mapped as: 1-2 = Negative, 3 = Neutral, 4-5 = Positive.

In [3]:
import sys
sys.path.append(str(PROJECT_ROOT))

from src.sentiment_analysis.data_loader import load_prepared_reviews

prepared_df = load_prepared_reviews(DATASET_PATH)
print(prepared_df[['title', 'rating', 'sentiment', 'review_text']].head().to_string(index=False))
print('\nSentiment counts:')
print(prepared_df['sentiment'].value_counts().to_string())

                                  title  rating sentiment                                                                                                                                                                                                                                                                                                                                                                                                           review_text
                       Horrible product       1  Negative                                                                                                                                                                                                                                                                                                                                          horrible product very disappointed with the overall performance from samsung
Camera quality is not like 48 megapixel       3   Neutral               

## 3. Train Model

In [4]:
from src.sentiment_analysis.train_utils import train_sentiment_model, save_model
from src.sentiment_analysis.evaluation import save_metrics
from src.sentiment_analysis.config import (
    DEFAULT_CLASSIFICATION_REPORT_PATH,
    DEFAULT_CONFUSION_MATRIX_PATH,
    DEFAULT_METRICS_PATH,
    DEFAULT_MODEL_PATH,
    DEFAULT_PROCESSED_PATH,
)

model, metrics, split_info, prepared_df = train_sentiment_model(DATASET_PATH)
save_model(model, DEFAULT_MODEL_PATH)
save_metrics(metrics, DEFAULT_METRICS_PATH, DEFAULT_CLASSIFICATION_REPORT_PATH, DEFAULT_CONFUSION_MATRIX_PATH)
DEFAULT_PROCESSED_PATH.parent.mkdir(parents=True, exist_ok=True)
prepared_df.to_csv(DEFAULT_PROCESSED_PATH, index=False)

print('Training completed successfully.')
print(split_info)
print(f"Accuracy: {metrics['accuracy']:.4f}")
print(f"Macro F1: {metrics['macro_f1']:.4f}")
print(f"Weighted F1: {metrics['weighted_f1']:.4f}")

Training completed successfully.
{'train_rows': 1152, 'test_rows': 288, 'total_rows': 1440, 'class_distribution': {'Positive': 729, 'Negative': 512, 'Neutral': 199}}
Accuracy: 0.7569
Macro F1: 0.6400
Weighted F1: 0.7520


## 4. Evaluation Report

In [5]:
report_df = pd.read_csv(DEFAULT_CLASSIFICATION_REPORT_PATH, index_col=0)
confusion_df = pd.read_csv(DEFAULT_CONFUSION_MATRIX_PATH, index_col=0)
print('Classification report:')
print(report_df.round(3).to_string())
print('\nConfusion matrix:')
print(confusion_df.to_string())

Classification report:
              precision  recall  f1-score  support
Negative          0.798   0.814     0.806  102.000
Neutral           0.286   0.250     0.267   40.000
Positive          0.839   0.856     0.847  146.000
accuracy          0.757   0.757     0.757    0.757
macro avg         0.641   0.640     0.640  288.000
weighted avg      0.748   0.757     0.752  288.000

Confusion matrix:
          Negative  Neutral  Positive
Negative        83       11         8
Neutral         14       10        16
Positive         7       14       125


## 5. Sample Prediction

In [6]:
from src.sentiment_analysis.predict import predict_sentiment

sample_review = 'Battery life is excellent and the phone performance is smooth.'
print(predict_sentiment(sample_review, model))

{'sentiment': 'Positive', 'cleaned_text': 'battery life is excellent and the phone performance is smooth', 'probabilities': {'Negative': 0.0063, 'Neutral': 0.0136, 'Positive': 0.9801}, 'confidence': 0.9801}
